In [ ]:
# BY GPT 

# 빅카인즈 뉴스기사 크롤링
# Step 1. 필요한 라이브러리 로딩
import os
import time
import random
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    NoSuchElementException,
    TimeoutException,
    StaleElementReferenceException,
    ElementClickInterceptedException,
    ElementNotInteractableException
)

# -----------------------------------
# 공통 함수
# -----------------------------------
def safe_find_text(parent, by, selector, default=""):
    try:
        return parent.find_element(by, selector).text.strip()
    except:
        return default

def close_start_popups(driver):
    """
    처음 사이트 접속 시 뜨는 팝업 닫기
    있으면 닫고, 없으면 그냥 진행
    """
    try:
        time.sleep(2)
        popups = driver.find_elements(By.CSS_SELECTOR, 'button[data-name="popup-dismiss"]')
        for popup in popups:
            try:
                driver.execute_script("arguments[0].click();", popup)
                time.sleep(1)
            except:
                pass
    except:
        pass

def close_news_detail_popup(driver):
    """
    기사 상세 팝업 닫기
    여러 방식으로 시도
    """
    time.sleep(1)

    # 1순위: 하단 '닫기' 버튼
    try:
        close_btn = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, '#news-detail-modal .modal-footer button[data-dismiss="modal"]')
            )
        )
        driver.execute_script("arguments[0].click();", close_btn)
        time.sleep(1)
        return True
    except:
        pass

    # 2순위: 우상단 X 버튼
    try:
        close_btn = driver.find_element(By.CSS_SELECTOR, '#news-detail-modal button.modal-close')
        driver.execute_script("arguments[0].click();", close_btn)
        time.sleep(1)
        return True
    except:
        pass

    # 3순위: ESC
    try:
        body = driver.find_element(By.TAG_NAME, "body")
        body.send_keys(Keys.ESCAPE)
        time.sleep(1)
        return True
    except:
        pass

    return False

def wait_until_popup_closed(driver):
    try:
        WebDriverWait(driver, 10).until(
            EC.invisibility_of_element_located((By.ID, "news-detail-modal"))
        )
        time.sleep(1)
        return True
    except:
        return False

def go_next_page(driver):
    """
    다음 페이지 버튼 클릭
    """
    time.sleep(1)

    # 방법 1: JS 클릭
    try:
        next_btn = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "a.page-next.page-link"))
        )
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", next_btn)
        time.sleep(1)
        driver.execute_script("arguments[0].click();", next_btn)
        time.sleep(2)
        return True
    except:
        pass

    # 방법 2: 일반 클릭
    try:
        next_btn = driver.find_element(By.CSS_SELECTOR, "a.page-next.page-link")
        next_btn.click()
        time.sleep(2)
        return True
    except:
        pass

    return False

def extract_article_meta(article):
    """
    목록에서 기사 메타정보 추출
    """
    news_id = article.get_attribute("data-id")

    title = safe_find_text(article, By.CSS_SELECTOR, ".title-elipsis", "")
    summary = safe_find_text(article, By.CSS_SELECTOR, "p.text", "")

    names = article.find_elements(By.CSS_SELECTOR, ".info p.name")
    date = names[0].text.strip() if len(names) > 0 else ""
    reporter = names[1].text.strip() if len(names) > 1 else ""

    media = ""
    try:
        media = article.find_element(By.CSS_SELECTOR, ".info a.provider").text.strip()
    except:
        try:
            media = article.find_element(By.CSS_SELECTOR, ".info > div").text.strip()
        except:
            media = ""

    return {
        "news_id": news_id,
        "title": title,
        "summary": summary,
        "date": date,
        "reporter": reporter,
        "media": media
    }

def open_article_and_get_content(driver, article):
    """
    기사 클릭 후 본문 추출
    """
    # 기사 클릭
    detail_btn = article.find_element(By.CSS_SELECTOR, "a.news-detail")
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", detail_btn)
    time.sleep(1)
    driver.execute_script("arguments[0].click();", detail_btn)

    # 팝업 대기
    WebDriverWait(driver, 10).until(
        EC.visibility_of_element_located((By.ID, "news-detail-modal"))
    )
    time.sleep(1)

    # 부제목
    try:
        subtitle = driver.find_element(By.CSS_SELECTOR, ".news-view-body h5").text.strip()
    except:
        subtitle = ""

    # 본문
    try:
        content = driver.find_element(By.CSS_SELECTOR, ".news-view-content").text.strip()
    except:
        content = ""

    content = content.replace("\n", " ").strip()

    if subtitle != "":
        article_text = subtitle + " " + content
    else:
        article_text = content

    return subtitle, content, article_text


# -----------------------------------
# Step 2. 사용자 입력
# -----------------------------------
print("=" * 80)
print("빅카인즈 사이트의 뉴스기사 추출하기")
print("=" * 80)

f_dir = input("1. 파일을 저장할 폴더명만 쓰세요(기본경로:c:\\py_temp\\): ").strip()
if f_dir == "":
    f_dir = "c:\\py_temp\\"

query_txt = input("2. 크롤링할 뉴스의 키워드는 무엇입니까?: ").strip()

start_date = input("3. 시작날짜를 입력하세요 (예: 2026-01-01, 엔터 시 기본값 사용): ").strip()
end_date = input("4. 종료날짜를 입력하세요 (예: 2026-03-12, 엔터 시 기본값 사용): ").strip()

include_words = input("5. 단어 중 1개 이상 포함할 단어를 입력하세요(없으면 엔터): ").strip()
exclude_words = input("6. 제외할 단어를 입력하세요(없으면 엔터): ").strip()

print("\n정렬 기준 선택")
print("1. 최신순")
print("2. 정확도순")
print("3. 과거순")
sort_input = input("7. 번호 입력: ").strip()

print("\n기사 표시 개수 선택")
print("1. 10건")
print("2. 30건")
print("3. 50건")
print("4. 100건")
count_input = input("8. 번호 입력: ").strip()

# count_input -> 실제 per_page 숫자로 변환
count_dict = {
    "1": 10,
    "2": 30,
    "3": 50,
    "4": 100
}
per_page = count_dict.get(count_input, 10)

sort_dict = {
    "1": "date",
    "2": "score",
    "3": "past"
}
sort_value = sort_dict.get(sort_input, "date")

# -----------------------------------
# Step 3. 크롬 드라이버 설정
# -----------------------------------
s_time = time.time()

s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)
driver.maximize_window()

driver.get("https://www.bigkinds.or.kr/")
time.sleep(random.randrange(2, 5))

# -----------------------------------
# Step 4. 시작 팝업 닫기
# -----------------------------------
close_start_popups(driver)

# -----------------------------------
# Step 5. 검색어 입력
# -----------------------------------
search_box = driver.find_element(By.ID, "total-search-key")
search_box.click()
time.sleep(1)
search_box.send_keys(Keys.CONTROL, "a")
search_box.send_keys(Keys.DELETE)
search_box.send_keys(query_txt)
time.sleep(1)

# -----------------------------------
# Step 6. 상세검색 열기
# -----------------------------------
detail_btn = driver.find_element(By.ID, "ig-sd-btn")
detail_btn.click()
time.sleep(2)

# 기간 탭 클릭
period_tab = driver.find_element(By.CSS_SELECTOR, "a.tab-btn.btn1")
period_tab.click()
time.sleep(1)

# 시작일 입력
if start_date != "":
    start_box = driver.find_element(By.ID, "search-begin-date")
    start_box.click()
    start_box.send_keys(Keys.CONTROL, "a")
    start_box.send_keys(Keys.DELETE)
    start_box.send_keys(start_date)
    time.sleep(1)

# 종료일 입력
if end_date != "":
    end_box = driver.find_element(By.ID, "search-end-date")
    end_box.click()
    end_box.send_keys(Keys.CONTROL, "a")
    end_box.send_keys(Keys.DELETE)
    end_box.send_keys(end_date)
    time.sleep(1)

# 상세검색 탭 클릭
detail_tab = driver.find_element(By.CSS_SELECTOR, "a.tab-btn.btn5")
detail_tab.click()
time.sleep(1)

# 포함어 입력
if include_words != "":
    try:
        or_box = driver.find_element(By.ID, "orKeyword1")
        or_box.click()
        or_box.send_keys(Keys.CONTROL, "a")
        or_box.send_keys(Keys.DELETE)
        or_box.send_keys(include_words)
        time.sleep(1)
    except:
        pass

# 제외어 입력
if exclude_words != "":
    try:
        not_box = driver.find_element(By.ID, "notKeyword1")
        not_box.click()
        not_box.send_keys(Keys.CONTROL, "a")
        not_box.send_keys(Keys.DELETE)
        not_box.send_keys(exclude_words)
        time.sleep(1)
    except:
        pass

# 적용하기 클릭
apply_btn = driver.find_element(By.CSS_SELECTOR, "button.apply-btn.primary-btn.news-search-btn")
driver.execute_script("arguments[0].click();", apply_btn)
time.sleep(3)

# -----------------------------------
# Step 7. 검색 결과 확인
# -----------------------------------
search_keyword = driver.find_element(By.CLASS_NAME, "total-search-key").text
total_count_text = driver.find_element(By.CLASS_NAME, "total-news-cnt").text
start_date_text = driver.find_element(By.CLASS_NAME, "start-date-key").text
end_date_text = driver.find_element(By.CLASS_NAME, "end-date-key").text

total_count = int(total_count_text.replace(",", ""))

print("=" * 80)
print("검색 조건 확인")
print("=" * 80)
print("검색어 :", search_keyword)
print("검색 기간 :", start_date_text, "~", end_date_text)
print("검색 결과 :", f"{total_count:,}건")
print("=" * 80)

if total_count == 0:
    print("검색 결과가 0건입니다. 프로그램을 종료합니다.")
    driver.quit()
    raise SystemExit

# -----------------------------------
# Step 8. 수집 건수 입력
# -----------------------------------
collect_count = input("9. 몇 건의 기사를 크롤링할까요? (엔터 또는 0 입력 시 50건): ").strip()

if collect_count == "" or collect_count == "0":
    target_count = 50
else:
    target_count = int(collect_count)

if target_count > total_count:
    target_count = total_count

print(f"최종 수집 건수 : {target_count:,}건")

# -----------------------------------
# Step 9. 정렬/표시개수 설정
# -----------------------------------
try:
    sort_box = Select(driver.find_element(By.ID, "select1"))
    sort_box.select_by_value(sort_value)
    time.sleep(2)
except:
    pass

try:
    count_box = Select(driver.find_element(By.ID, "select2"))
    count_box.select_by_value(str(per_page))
    time.sleep(2)
except:
    pass

print(f"정렬 기준 : {sort_value}")
print(f"페이지당 표시 개수 : {per_page}")

# -----------------------------------
# Step 10. 기사 수집 시작
# -----------------------------------
results = []
collected_count = 0
current_page = 1

while collected_count < target_count:
    time.sleep(2)

    articles = driver.find_elements(By.CSS_SELECTOR, "#news-results .news-item")
    page_article_count = len(articles)

    if page_article_count == 0:
        print("현재 페이지에서 기사 목록을 찾지 못했습니다. 종료합니다.")
        break

    print("=" * 80)
    print(f"현재 페이지 : {current_page}")
    print(f"현재 페이지 기사 수 : {page_article_count}")
    print(f"현재까지 수집 건수 : {collected_count}")
    print("=" * 80)

    for i in range(page_article_count):
        if collected_count >= target_count:
            break

        try:
            # 기사 목록은 매번 다시 가져와야 안전함
            articles = driver.find_elements(By.CSS_SELECTOR, "#news-results .news-item")
            article = articles[i]

            meta = extract_article_meta(article)

            print(f"[{collected_count + 1}/{target_count}] {meta['title']}")

            subtitle, content, article_text = open_article_and_get_content(driver, article)

            meta["subtitle"] = subtitle
            meta["content"] = article_text

            results.append(meta)
            collected_count += 1

            # 팝업 닫기
            close_ok = close_news_detail_popup(driver)
            if not close_ok:
                print("상세 팝업 닫기 실패")
                break

            wait_until_popup_closed(driver)
            time.sleep(1)

        except StaleElementReferenceException:
            print(f"{i+1}번째 기사: stale element 발생 -> 다음 기사로 넘어감")
            continue

        except TimeoutException:
            print(f"{i+1}번째 기사: 팝업 로딩 시간 초과 -> 다음 기사로 넘어감")
            try:
                close_news_detail_popup(driver)
                wait_until_popup_closed(driver)
            except:
                pass
            continue

        except Exception as e:
            print(f"{i+1}번째 기사 처리 중 오류 발생: {e}")
            try:
                close_news_detail_popup(driver)
                wait_until_popup_closed(driver)
            except:
                pass
            continue

    # 목표 건수 채우면 종료
    if collected_count >= target_count:
        break

    # 다음 페이지 이동
    moved = go_next_page(driver)
    if not moved:
        print("다음 페이지 이동 실패. 수집을 종료합니다.")
        break

    current_page += 1

# -----------------------------------
# Step 11. 파일 저장
# -----------------------------------
df = pd.DataFrame(results)

if len(df) > 0:
    os.makedirs(f_dir, exist_ok=True)

    csv_name = os.path.join(f_dir, "bigkinds_news.csv")
    xlsx_name = os.path.join(f_dir, "bigkinds_news.xlsx")

    df.to_csv(csv_name, index=False, encoding="utf-8-sig")
    df.to_excel(xlsx_name, index=False)

    print("\n" + "=" * 80)
    print(f"총 {len(df)}건 수집 완료")
    print("CSV 저장 경로 :", csv_name)
    print("엑셀 저장 경로 :", xlsx_name)
    print("=" * 80)
else:
    print("저장할 데이터가 없습니다.")

# -----------------------------------
# Step 12. 종료
# -----------------------------------
e_time = time.time()
print(f"\n총 소요시간 : {round(e_time - s_time, 2)}초")

driver.quit()

빅카인즈 사이트의 뉴스기사 추출하기

정렬 기준 선택
1. 최신순
2. 정확도순
3. 과거순

기사 표시 개수 선택
1. 10건
2. 30건
3. 50건
4. 100건
검색 조건 확인
검색어 : 전쟁 AND (트럼프 OR 미국) NOT(김정은)
검색 기간 : 2026-03-03 ~ 2026-03-13
검색 결과 : 12,607건
최종 수집 건수 : 50건
정렬 기준 : date
페이지당 표시 개수 : 10
현재 페이지 : 1
현재 페이지 기사 수 : 10
현재까지 수집 건수 : 0
[1/50] 봉기 기대한 미·이스라엘의 오판…네타냐후마저 회의론[美-이란 전쟁]
[2/50] 주유소 곳곳 기름값 뚝…도심 외곽은 1700원대 등장도[르포]
[3/50] 국회 문턱 넘은 대미투자법…K-조선 ‘존스법 장벽’ 넘어 美 진출 물꼬 트나
[4/50] "호르무즈에 이미 기뢰 10개 깔렸다"…美당국 보고서 부인한 트럼프, 왜?
[5/50] ‘홀인원’에 겹친 이란 공습… 백악관, ‘전쟁 희화화’ 비판 거세
[6/50] 판 뒤집을 아이디어 나왔다...'호르무즈 대안' 급부상 [지금이뉴스]
[7/50] [종합] 이란 새 최고지도자 첫 강경 메시지…트럼프도 "정권 완전 파괴" 맞불
[8/50] [송윤서의 '주'경야독] 지속되는 국제유가 상승…'롤러코스터' 코스피 전망은
[9/50] 석유 최고가제 시행 첫날…전국 주유소 44% 기름값 내렸다
[10/50] 치솟는 기름값에 덜덜 … "중고차, 전기차로 살까"
현재 페이지 : 2
현재 페이지 기사 수 : 10
현재까지 수집 건수 : 10
[11/50] 수천명 숨진 끔찍한 전쟁이 게임? 폭격 장면에 “홀인원”…백악관 영상 ‘뭇매’
[12/50] [마감시황] 코스피, '외국인·기관' 매도에 1%대 하락...5500선 하회
[13/50] 호르무즈서 원료 수급 막히자…美 비료·유럽 정유 반사이익
[14/50] 美, 토마호크만 168발 소진…"日과 미사일 공동생산 추진할 듯"
[15/50] 올해 30% 오른 코스피, 재평가는 이제 시작
[16/50] 

In [ ]:
# BY GEMINNI 
import os
import time
import random
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select
import math

# 1. 초기 설정
print("=" * 80)
print(" 빅카인즈 뉴스기사 메가 크롤러 ")
print("=" * 80)

query_txt = input("1. 크롤링할 키워드는 무엇입니까?: ").strip()
collect_count = int(input("2. 수집할 총 건수는 몇 건입니까?: "))
count_input = input("3. 한 페이지당 표시 건수(10, 30, 50, 100): ").strip()

f_dir = input("4. 저장할 폴더명(기본 c:\\py_temp\\): ")
if f_dir == '': f_dir = "c:\\py_temp\\"
if not os.path.exists(f_dir): os.makedirs(f_dir)

# 드라이버 실행
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)
driver.get('https://www.bigkinds.or.kr/')
driver.maximize_window()
time.sleep(random.randrange(2, 4))

# 팝업 닫기
popups = driver.find_elements(By.CSS_SELECTOR, 'button[data-name="popup-dismiss"]')
for popup in popups:
    try: popup.click()
    except: pass

# 2. 검색어 입력 및 상세검색 설정
search_box = driver.find_element(By.ID, "total-search-key")
search_box.send_keys(query_txt)
driver.find_element(By.ID, "ig-sd-btn").click() # 상세검색 열기
time.sleep(1)

# [적용하기] 버튼 클릭 (기본 3개월 설정 사용)
apply_btn = driver.find_element(By.CSS_SELECTOR, "button.apply-btn.primary-btn.news-search-btn")
driver.execute_script("arguments[0].click();", apply_btn)
time.sleep(3)

# 3. 출력 개수 설정 (select2)
count_menu = Select(driver.find_element(By.ID, "select2"))
count_menu.select_by_value(count_input)
time.sleep(2)

# 4. 수집 시작
collected_data = []
collected_count = 0
per_page = int(count_input)

print(f"\n>>> 조회를 시작합니다. (목표: {collect_count}건)")

while collected_count < collect_count:
    # 현재 페이지의 기사 목록 가져오기
    articles = driver.find_elements(By.CSS_SELECTOR, ".news-item")
    
    for article in articles:
        if collected_count >= collect_count:
            break
            
        try:
            # 목록 정보 추출
            title = article.find_element(By.CSS_SELECTOR, ".title-elipsis").text.strip()
            names = article.find_elements(By.CSS_SELECTOR, ".info p.name")
            date = names[0].text.strip() if len(names) > 0 else ""
            reporter = names[1].text.strip() if len(names) > 1 else ""
            
            try:
                media = article.find_element(By.CSS_SELECTOR, ".info a.provider").text.strip()
            except:
                media = article.find_element(By.CSS_SELECTOR, ".info > div").text.strip()

            # 상세 팝업 열기 (제목 클릭)
            detail_btn = article.find_element(By.CSS_SELECTOR, "a.news-detail")
            driver.execute_script("arguments[0].click();", detail_btn)
            
            # 팝업 내용 대기 및 추출
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, "news-view-content")))
            time.sleep(1)
            
            try:
                subtitle = driver.find_element(By.CSS_SELECTOR, ".news-view-body h5").text.strip()
            except:
                subtitle = ""
            
            content = driver.find_element(By.CLASS_NAME, "news-view-content").text.replace("\n", " ").strip()
            
            # 데이터 저장
            collected_data.append({
                '번호': collected_count + 1,
                '날짜': date,
                '언론사': media,
                '기자': reporter,
                '제목': title,
                '소제목': subtitle,
                '본문': content
            })
            
            collected_count += 1
            print(f"[{collected_count}/{collect_count}] {title[:30]}...")

            # 팝업 닫기 (하단 닫기 버튼 강제 클릭)
            close_btn = driver.find_element(By.CSS_SELECTOR, '#news-detail-modal .modal-footer button[data-dismiss="modal"]')
            driver.execute_script("arguments[0].click();", close_btn)
            time.sleep(0.8)
            
        except Exception as e:
            print(f"기사 수집 중 오류 발생(스킵): {e}")
            continue

    # 다음 페이지 이동
    if collected_count < collect_count:
        try:
            next_btn = driver.find_element(By.CSS_SELECTOR, "a.page-next.page-link")
            driver.execute_script("arguments[0].click();", next_btn)
            print("\n>>> 다음 페이지로 이동 중...\n")
            time.sleep(3)
        except:
            print("\n>>> 더 이상 페이지가 없습니다. 수집을 종료합니다.")
            break

# 5. 결과 저장
df = pd.DataFrame(collected_data)
file_name = f"bigkinds_{query_txt}_{time.strftime('%Y%m%d_%H%M%S')}.xlsx"
full_path = os.path.join(f_dir, file_name)
df.to_excel(full_path, index=False)

print("=" * 80)
print(f"수집 완료! 총 {len(df)}건의 기사가 저장되었습니다.")
print(f"파일 경로: {full_path}")
print("=" * 80)

driver.quit()

 빅카인즈 뉴스기사 메가 크롤러 

>>> 조회를 시작합니다. (목표: 60건)
[1/60] 유가 급등에 환율 야간 거래 장중 1500원 또 넘어...
[2/60] 고환율에 수익률 희비…환노출 ETF 웃는다...
[3/60] 환율 1500원 재돌파...중동사태 장기화 우려에 맥못...
[4/60] 환율, 유가 급등에 이틀째 올라…야간거래 장중 1,50...
[5/60] 국제유가 고공행진…환율 또 1500원 터치 [한경 외환...
[6/60] 유가 급등에 달러인덱스 100선 넘었다...
[7/60] [속보] 원·달러 환율, 야간 거래에서 1500원 돌파...
[8/60] [속보]원·달러 환율 야간 거래 장중 1500원 또 돌...
[9/60] [속보] 원/달러 환율 야간 거래 장중 1500원 또 ...
[10/60] 서진시스템, 지난해 영업익 11억4578만원...전년비...
[11/60] [속보]원·달러 환율 야간 거래 장중 1500원 재돌파...
[12/60] [동십자각] 상생 외쳤는데 ‘갈등만 빗발’...
[13/60] [속보] 원·달러 환율 야간 거래 장중 1500원 또 ...
[14/60] 유가 쇼크에…원·달러 환율 야간 거래서 1500원 돌파...
[15/60] [속보] 원·달러 환율, 야간 거래에서 1,500원 돌...
[16/60] [속보] 원·달러 환율, '야간 거래' 장중 1500원...
[17/60] [송윤서의 '주'경야독] 지속되는 국제유가 상승…'롤러...
[18/60] [마감시황] 코스피, '외국인·기관' 매도에 1%대 하...
[19/60] [속보] 원·달러 환율 야간 거래 장중 1500원 또 ...
[20/60] [속보] 원/달러 환율 야간 거래 장중 1,500원 또...
[21/60] [간추린 경제] 1. 1,500원 다시 위협 2. 다시...
[22/60] [속보] 원·달러 환율 야간 거래 장중 1500원 넘겨...
[23/60] 외국인·기관 2.5조 순매도에 주저앉은 코스피...
[24/60] 트럼프 "오늘 밤 지켜보라"...중